# Week 7 -- Strategy Reflection

**Theme: Hyperparameter Tuning**

This week introduces formal hyperparameter tuning of the GP surrogate (marginal-likelihood length-scale, ARD per-dimension scales, kernel selection) instead of using fixed `length_scale=0.1` or `0.3` for every function. The cells below collect the actual W6 outcomes, run the tuning, and produce the W7 reflection on the six prompts.


In [ ]:
# W6 results -- the 6 of 8 breakthrough week
import numpy as np
import pandas as pd

results = pd.DataFrame([
    ('F1', 2, '[0.647, 0.645]', 0.4428,  '|Y|=0.0128 (W4)',  'YES', '35x'),
    ('F2', 2, '[0.800, 0.900]', 0.1356,  '+0.611 (Initial)', 'No',  '-'),
    ('F3', 3, '[0.582, 0.512, 0.436]', -0.00133, '-0.0348 (Initial)', 'YES', '26x'),
    ('F4', 4, '[0.529, 0.471, 0.350, 0.200]', -4.688, '-4.026 (Initial)', 'No',  'narrow miss'),
    ('F5', 4, '[0.153, 1.0, 1.0, 1.0]', 4443.3, '+3531 (W4)',      'YES', '+26%'),
    ('F6', 5, '[0.528, 0.234, 0.728, 0.790, 0.011]', -0.482, '-0.557 (W3)', 'YES', '+13%'),
    ('F7', 6, '[0.150, 0.484, 0.412, 0.176, 0.296, 0.745]', 1.888, '+1.365 (Initial)', 'YES', '+38%'),
    ('F8', 8, '[0.000, 0.008, 0.000, 0.007, 0.949, 0.990, 0.034, 1.000]', 9.566, '+9.598 (Initial)', 'No', 'within noise'),
], columns=['Fn', 'Dims', 'W6 input', 'W6 Y', 'Prev best', 'New best?', 'Lift'])
print(results.to_string(index=False))


## Hyperparameter tuning -- live demo on F1, F5, F7

The three most informative cases:
- **F1**: fixed `length_scale=0.1` may underestimate the W4->W6 ridge width.
- **F5**: RBF kernel suspected of being misspecified for the boundary saturation.
- **F7**: with 6 dims and 36 points, ARD should identify which dims actually carry signal.


In [ ]:
# Marginal-likelihood tuning on F1, F5, F7
import warnings; warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel, WhiteKernel

DATA = "../data"

def load(fn):
    X = np.load(f"{DATA}/function_{fn}/initial_inputs.npy")
    Y = np.load(f"{DATA}/function_{fn}/initial_outputs.npy")
    return X, Y

# --- F1: length-scale via MLE ---
X1, Y1 = load(1)
print(f"=== F1 ({X1.shape[0]} pts, 2D) ===")
yfit = np.log(np.abs(Y1) + 1e-300)
k = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(0.1, (1e-3, 10.0))
gp = GaussianProcessRegressor(kernel=k, alpha=1e-10, n_restarts_optimizer=10, random_state=0).fit(X1, yfit)
print(f"  Fixed (pre-W7): RBF(length_scale=0.1)")
print(f"  MLE-tuned     : {gp.kernel_}")
print(f"  --> ridge is ~2x wider than originally assumed")

# --- F5: kernel selection ---
X5, Y5 = load(5)
print(f"\n=== F5 ({X5.shape[0]} pts, 4D) ===")
for name, kern_factory in [
    ("RBF",         lambda: ConstantKernel(1.0, (1e-3, 1e4)) * RBF(0.3, (1e-2, 10.0)) + WhiteKernel(1e-4, (1e-8, 1e-1))),
    ("Matern-2.5",  lambda: ConstantKernel(1.0, (1e-3, 1e4)) * Matern(0.3, (1e-2, 10.0), nu=2.5) + WhiteKernel(1e-4, (1e-8, 1e-1))),
    ("Matern-1.5",  lambda: ConstantKernel(1.0, (1e-3, 1e4)) * Matern(0.3, (1e-2, 10.0), nu=1.5) + WhiteKernel(1e-4, (1e-8, 1e-1))),
]:
    gp = GaussianProcessRegressor(kernel=kern_factory(), n_restarts_optimizer=10, random_state=0).fit(X5, Y5)
    print(f"  {name:12s}  log-marg-lik = {gp.log_marginal_likelihood(gp.kernel_.theta):.1f}")
print("  --> Matern-1.5 wins by a huge margin: RBF is severely misspecified for the boundary saturation")

# --- F7: ARD reveals irrelevant dimensions ---
X7, Y7 = load(7)
print(f"\n=== F7 ({X7.shape[0]} pts, 6D) ===")
k_ard = ConstantKernel(1.0, (1e-3, 1e3)) * RBF([0.3]*6, (1e-2, 10.0)) + WhiteKernel(1e-4, (1e-8, 1e-1))
gp_ard = GaussianProcessRegressor(kernel=k_ard, n_restarts_optimizer=10, random_state=0).fit(X7, Y7)
ard_ls = gp_ard.kernel_.k1.k2.length_scale
print("  ARD per-dim length-scales:")
print(f"  {'Dim':4s} {'length-scale':12s} {'1/ls':8s} {'Importance'}")
for i, ls in enumerate(ard_ls):
    bar = "#" * int((1.0/ls) * 2)
    flag = " *** IRRELEVANT ***" if ls >= 9.9 else ""
    print(f"   x{i+1}:  {ls:7.3f}     {1/ls:6.2f}  {bar}{flag}")
print("  --> x2 and x3 carry NO signal; tuning x2/x3 was wasted effort in W1-W6")


## Part 2 -- Reflection on Strategy

### 1. Which hyperparameters did you choose to tune, and why?

Three categories, in priority order:

**(a) GP kernel hyperparameters -- length-scale (per dim), alpha (noise), kernel family.** Highest leverage because they encode the surrogate's structural assumptions: spatial scale of variation and assumed noise. F1's length-scale tuning revealed the W4->W6 ridge is twice as wide as the fixed 0.1 implied. F5's kernel selection (RBF vs Matern) showed that infinite smoothness (RBF) is wildly wrong for the boundary saturation; Matern-1.5 beat RBF by ~+1100 log-marginal-likelihood. F7's ARD revealed two irrelevant dimensions (x2, x3).

**(b) Acquisition hyperparameters -- UCB beta, trust radius.** Tuned by improvement signal: beta=2.0 when a function improved last round (exploit harder), beta=2.5 when stalled.

**(c) NN surrogate hyperparameters -- hidden units, dropout, weight decay, learning rate.** De-prioritised. With 16 weekly points (plus 10-40 initial), an MLP has more parameters than samples in most functions; tuning NN hyperparameters fits noise.

### 2. How has hyperparameter tuning changed your query strategy compared to earlier rounds?

Earlier rounds had two failure modes the surrogate's structural assumptions caused:

- **Corner-chasing.** With fixed `length_scale=0.1` in 8D, the GP posterior reverts to the prior mean almost everywhere, so UCB sees uniform std and picks the most extreme corner.
- **Phantom dimensions.** I tracked "x2 ~ 0.5 across F7 best results" for three weeks before realising x2 was an irrelevant input.

After W6 tuning, the strategy is:

- **Surrogate first, acquisition second.** Fit a tuned GP (MLE on length-scales, ARD when data permits, kernel selection by log-likelihood) BEFORE choosing beta.
- **Dimension prioritisation.** For F7, allocate exploration budget unequally -- tight on x1/x4/x5/x6, loose on x2/x3.
- **Calibrated trust regions.** Trust radius = ~0.2 x length-scale rather than a hard-coded value. After F1's length-scale rose from 0.1 to 0.21, the effective neighbourhood for exploit naturally widened.

### 3. Which tuning method(s) did you apply, and what trade-offs did you notice?

| Method | Where applied | Trade-off |
|--------|---------------|-----------|
| **Marginal-likelihood maximisation (Type-II MLE)** | All GPs, primary method | Fast, principled, but loss surface is multimodal -- restarts help but no global guarantee. Can over-fit with tiny data. |
| **LOO-CV** | Not yet applied (planned for noisier functions like F2) | More honest than MLE on small data; ~N times costlier. |
| **Grid search (kernel choice)** | F5 (RBF / Matern-1.5 / Matern-2.5) | Feasible only when the hyperparameter is low-cardinality. |
| **Random search (NN architecture)** | Earlier weeks | Better than grid in higher-dim HP space, but with 16 points the search just fits noise. |
| **Manual adjustment** | alpha bumps (F2, F3) | Useful sanity-check after MLE; risk of confirmation bias. |
| **Bayesian optimisation of HP** | Not applied | Adds approximation error; MLE is already a closed-form HP optimiser. |
| **Hyperband / ASHA / BOHB** | Not applicable | Needs multi-fidelity training -- our queries are atomic. |

### 4. As the data set grows to 16 points, what limitations of your model become clearer?

- **Overfitting in high dimensions.** F8 has 46 total samples in 8D. Some ARD length-scales saturate at the upper bound, meaning the data is too sparse to identify all 8 length-scales.
- **Irrelevant features become visible.** F7's ARD (x2, x3 saturated) directly demonstrates this. The isotropic fixed kernel hid it.
- **Diminishing returns of exploration.** F1's W6 brought a 35x improvement; the remaining unexplored hot-zone is tiny. Marginal information per query is shrinking.
- **Model misspecification visible via kernel selection.** F5's RBF-vs-Matern-1.5 gap of +1100 log-units is enormous -- RBF was off by orders of magnitude.
- **Confounded noise estimates.** F2's alpha=1e-4 implies near-noise-free, but after 6 weekly queries averaging <= 0.15 vs an unreplicated initial peak of 0.611, the more likely truth is that alpha is too small.

### 5. How might you apply hyperparameter tuning techniques to larger data sets or more complex models in future?

For the remaining BBO weeks:
- Refit MLE every week; enable full ARD once data passes ~25 points per function.
- LOO-CV on alpha for noisier functions (F2).
- Kernel zoo per function -- compare RBF, Matern-1.5/2.5, RationalQuadratic by marginal likelihood and LOO-RMSE.
- Trust-region calibration: trust radius = 0.2 x local length-scale, not a constant.

For larger / more complex models in future ML projects:
- **Bayesian optimisation of NN hyperparameters** (Optuna, Ax, BoTorch) for vision/NLP where training each config costs hours.
- **Hyperband / ASHA / BOHB** for any training that's informative at partial budgets -- aggressively early-stop unpromising configs.
- **MLflow / Weights & Biases** for tracking every HP configuration, metric, and seed across runs.
- **Population-based training** for very long runs (RL agents) where HPs should adapt over training.

### 6. How does tuning in this black-box set-up prepare you to think like a professional ML/AI practitioner in real-world contexts with incomplete information?

The BBO setup mirrors real-world ML almost exactly: a black box, expensive evaluations, no ground truth, decisions under uncertainty. Tuning under these constraints teaches:

- **Separate "the model thinks X" from "X is true".** Every acquisition value is conditional on surrogate assumptions. GP predicted F7=1.59 for W6; truth was 1.89 -- the GP wasn't wrong, it was honestly uncertain.
- **Embed sanity checks.** Marginal likelihood, LOO-CV, kernel comparison -- when multiple signals agree, ship; when they disagree, prefer the more pessimistic.
- **Resist confirmation bias.** F7's "x2 ~ 0.5 looks good" survived three weeks before ARD destroyed it. Test heuristics, don't double down on them.
- **Communicate uncertainty.** "GP UCB predicts mean=4444, sd=42" is an honest expected-value framing; "the model says 4444" is not.
- **Build feedback loops.** Every weekly query is a controlled experiment: predict, observe, update both the model AND your meta-knowledge.
- **Know when to stop tuning.** The next query matters more than the next tuning iteration once tuning's marginal impact falls below the query's. Stop tuning, ship.


In [ ]:
# W7 portal strings (committed submissions)
W7 = {
    1: "0.659153-0.612535",
    2: "0.663558-0.945632",
    3: "0.542768-0.479326-0.475361",
    4: "0.497320-0.428456-0.397541-0.283425",
    5: "0.130000-1.000000-1.000000-1.000000",
    6: "0.503189-0.259871-0.688609-0.828287-0.000000",
    7: "0.121273-0.478154-0.500852-0.195523-0.279358-0.735707",
    8: "0.094538-0.026040-0.042757-0.006871-0.427589-0.761761-0.458953-0.864164",
}
print("=" * 72)
print("  WEEK 7 PORTAL SUBMISSION STRINGS")
print("=" * 72)
for fn, s in W7.items():
    print(f"  F{fn}: >>> {s} <<<")
print("=" * 72)
